# Process stop-level delay data for the dashboard

Reads `stop_metrics_weekday.csv` and `stop_metrics_weekend.csv`, then writes two JSON files used by `js/delay.js`:

- **`network_routes.json`** &mdash; one entry per route: ordered stop sequence with coordinates and weighted-mean delay / on-time percentage per period (used for the network overview map, KPIs and route ranking).
- **`route_details.json`** &mdash; per-route, per-stop period summaries plus a small sample of raw delay observations (used for the route-detail tab: scatter plot and heatmap).

To keep the payload small enough to ship to the browser, only the top `N_ROUTES` routes (by total weekday arrivals) are exported, and at most `MAX_TRIPS_PER_PERIOD` raw delay observations are kept per stop / period.

In [1]:
import ast
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook lives in `data/delay/`; all paths are relative to that.
DATA_DIR = Path('.').resolve()
STOP_DIR = DATA_DIR / 'stop_metrics'
OUT_DIR  = DATA_DIR

N_ROUTES             = 40   # top-N routes by weekday arrivals
MIN_STOPS_PER_ROUTE  = 4    # drop routes with too few unique stops
MAX_TRIPS_PER_PERIOD = 12   # raw delay samples kept per stop / period

PERIOD_COLS = {
    'all':     'day',
    'am_off':  'morning_offpeak',
    'am_peak': 'morning_peak',
    'midday':  'midday_offpeak',
    'pm_peak': 'evening_peak',
    'pm_off':  'evening_offpeak',
}
DETAIL_PERIODS = [k for k in PERIOD_COLS if k != 'all']  # scatter / heatmap drop the 'all' bucket

In [2]:
df_wd = pd.read_csv(STOP_DIR / 'stop_metrics_weekday.csv')
df_we = pd.read_csv(STOP_DIR / 'stop_metrics_weekend.csv')
print('weekday rows:', len(df_wd), '· weekend rows:', len(df_we))
print('unique routes (weekday):', df_wd['route_short_name'].nunique())
print('unique stops  (weekday):', df_wd['stop_id'].nunique())
df_wd.head(2)

weekday rows: 51890 · weekend rows: 32047
unique routes (weekday): 577
unique stops  (weekday): 12322


,route_id,route_short_name,stop_id,stop_name,lon,lat,easting,northing,n_arrivals_day,Q1_day,...,list_delay_evening_peak,n_arrivals_evening_offpeak,Q1_evening_offpeak,Q2_evening_offpeak,Q3_evening_offpeak,min_evening_offpeak,max_evening_offpeak,pct_on_time_evening_offpeak,pct_late_evening_offpeak,list_delay_evening_offpeak
0,58936,199,1800SG36501,Alders Road,-2.065287,53.362049,395752.6,385021.5,39,1.84,...,"[6.18, 0.36, 3.98, 7.85, 8.18, 7.5, 5.85, 0.4]",8,1.61,3.46,5.17,-0.47,6.07,12.5,87.5,"[3.23, 5.99, -0.47, 1.76, 6.07, 1.17, 3.69, 4.9]"
1,58936,199,1800SG36471,Andrews Lane,-2.064600,53.362133,395798.3,385030.8,42,2.43,...,"[10.12, 17.9, 12.7, 20.61, 8.18, 0.16, 62.74, ...",13,2.02,4.31,13.06,1.25,1446.77,0.0,100.0,"[1.36, 4.56, 4.31, 2.0, 3.59, 1.25, 9.04, 1446..."


In [3]:
# Pick the top N routes by weekday arrival volume.
route_volume = (df_wd.groupby('route_short_name')['n_arrivals_day']
                       .sum().sort_values(ascending=False))
top_routes = route_volume.head(N_ROUTES).index.tolist()
print('top routes selected:', top_routes)

top routes selected: ['192', '163', '36', '17', '471', '84', '135', '409', '201', '59', '43', '52', '37', '8', '610', '38', '18', '203', '50', '41', '219', '609', '607', '501', '582', '11', '142', '67', '83', '330', '524', '575', '20', '216', '350', '10', '76', '100', '112', '53']


### Stop ordering

The CSVs interleave outbound / inbound stops in arbitrary order, so for the route polyline we order stops greedily along the line:

1. Pick the pair of stops that are furthest apart &mdash; an approximate end-to-end of the route.
2. Starting from one endpoint, repeatedly visit the nearest unvisited stop.

This produces a single connected sequence good enough for an overview polyline; it is not a true shortest-path solver.

In [4]:
def order_stops(latlons: np.ndarray) -> list[int]:
    n = len(latlons)
    if n <= 2:
        return list(range(n))
    # Cosine correction so the lon/lat distances behave like km.
    pts = latlons.copy()
    pts[:, 1] *= math.cos(math.radians(float(pts[:, 0].mean())))
    # Endpoints = pair with greatest pairwise distance (vectorised).
    diff = pts[:, None, :] - pts[None, :, :]
    d2   = (diff ** 2).sum(-1)
    i, _ = np.unravel_index(np.argmax(d2), d2.shape)
    order, visited, cur = [int(i)], {int(i)}, int(i)
    for _ in range(n - 1):
        d_row = d2[cur].copy()
        for v in visited:
            d_row[v] = np.inf
        nxt = int(np.argmin(d_row))
        order.append(nxt)
        visited.add(nxt)
        cur = nxt
    return order

In [5]:
def weighted_period_stats(stops_df: pd.DataFrame, period_col: str) -> dict:
    """Arrival-weighted mean of Q2 delay (median per stop) and pct on-time."""
    n_col   = f'n_arrivals_{period_col}'
    q2_col  = f'Q2_{period_col}'
    otp_col = f'pct_on_time_{period_col}'
    s = stops_df.dropna(subset=[q2_col])
    if s.empty or s[n_col].sum() == 0:
        return {'mean': 0.0, 'otp': 0.0, 'n': 0}
    w = s[n_col].astype(float)
    return {
        'mean': float((s[q2_col] * w).sum() / w.sum()),
        'otp':  float((s[otp_col].fillna(0) * w).sum() / w.sum()),
        'n':    int(w.sum()),
    }

In [6]:
# Build per-route summary: ordered stop sequence + weighted-mean delay per period.
network = []
for short_name in top_routes:
    sub = df_wd[df_wd['route_short_name'] == short_name].copy()
    # Some stops appear under multiple route_ids (directions). Keep the busiest entry.
    sub = (sub.sort_values('n_arrivals_day', ascending=False)
              .drop_duplicates('stop_id'))
    if len(sub) < MIN_STOPS_PER_ROUTE:
        continue
    sub = sub.reset_index(drop=True)
    order = order_stops(sub[['lat', 'lon']].to_numpy(dtype=float))
    sub_ord = sub.iloc[order].reset_index(drop=True)

    network.append({
        'name': str(short_name),
        'stops': [
            {
                'id':   r['stop_id'],
                'name': r['stop_name'],
                'lat':  round(float(r['lat']), 5),
                'lon':  round(float(r['lon']), 5),
            }
            for _, r in sub_ord.iterrows()
        ],
        'periods': {k: weighted_period_stats(sub_ord, col)
                    for k, col in PERIOD_COLS.items()},
    })

print(f'wrote {len(network)} routes')
print('mean stops per route:', sum(len(r["stops"]) for r in network) / len(network))

wrote 40 routes
mean stops per route: 124.575


In [7]:
def parse_delay_list(raw):
    if pd.isna(raw):
        return []
    try:
        return [round(float(x), 2) for x in ast.literal_eval(raw)]
    except (ValueError, SyntaxError):
        return []

def even_sample(arr, k):
    if len(arr) <= k:
        return arr
    idx = np.linspace(0, len(arr) - 1, k).round().astype(int)
    return [arr[i] for i in idx]

def stop_period_record(row, period_col):
    n = row.get(f'n_arrivals_{period_col}', 0)
    if pd.isna(n) or n == 0:
        return {'mean': None, 'otp': None, 'n': 0, 'd': []}
    delays = even_sample(parse_delay_list(row.get(f'list_delay_{period_col}')),
                         MAX_TRIPS_PER_PERIOD)
    return {
        'mean': None if pd.isna(row[f'Q2_{period_col}']) else round(float(row[f'Q2_{period_col}']), 2),
        'otp':  None if pd.isna(row[f'pct_on_time_{period_col}']) else round(float(row[f'pct_on_time_{period_col}']), 1),
        'n':    int(n),
        'd':    delays,
    }

In [8]:
# Build per-stop, per-period detail used by the route-detail tab (scatter + heatmap).
details = {}
for r in network:
    sn = r['name']
    rows_wd = df_wd[df_wd['route_short_name'] == sn].drop_duplicates('stop_id').set_index('stop_id')
    rows_we = df_we[df_we['route_short_name'] == sn].drop_duplicates('stop_id').set_index('stop_id')
    by_stop = {}
    for stop in r['stops']:
        sid = stop['id']
        wd_row = rows_wd.loc[sid] if sid in rows_wd.index else None
        we_row = rows_we.loc[sid] if sid in rows_we.index else None
        by_stop[sid] = {
            'wd': {p: stop_period_record(wd_row, PERIOD_COLS[p]) for p in DETAIL_PERIODS}
                  if wd_row is not None else {p: {'mean': None, 'otp': None, 'n': 0, 'd': []} for p in DETAIL_PERIODS},
            'we': {p: stop_period_record(we_row, PERIOD_COLS[p]) for p in DETAIL_PERIODS}
                  if we_row is not None else {p: {'mean': None, 'otp': None, 'n': 0, 'd': []} for p in DETAIL_PERIODS},
        }
    details[sn] = by_stop

print('detail records for', len(details), 'routes')

detail records for 40 routes


In [9]:
# Write the two JSON files used by js/delay.js.
summary_path = OUT_DIR / 'network_routes.json'
details_path = OUT_DIR / 'route_details.json'

with summary_path.open('w', encoding='utf-8') as f:
    json.dump({'routes': network}, f, separators=(',', ':'))
with details_path.open('w', encoding='utf-8') as f:
    json.dump(details, f, separators=(',', ':'))

for p in (summary_path, details_path):
    print(f'{p.name}: {p.stat().st_size / 1024:.1f} KB')

network_routes.json: 384.7 KB
route_details.json: 4810.6 KB


In [10]:
# Quick sanity check on one route.
r = network[0]
print(f"route {r['name']} · {len(r['stops'])} stops")
for k, v in r['periods'].items():
    print(f"  {k:8s} mean={v['mean']:.2f} min  otp={v['otp']:.1f}%  n={v['n']}")

route 192 · 107 stops
  all      mean=1.24 min  otp=28.4%  n=145046
  am_off   mean=2.62 min  otp=36.6%  n=14328
  am_peak  mean=0.14 min  otp=36.0%  n=16264
  midday   mean=1.22 min  otp=27.4%  n=55455
  pm_peak  mean=2.01 min  otp=22.0%  n=28683
  pm_off   mean=1.65 min  otp=28.3%  n=30316
